This are the mapping rules

In [0]:
%sql
UPDATE dbw_taxrisk_dev.config.disclosure_terms
SET meaning = 'paid_or_in_kind'
WHERE country_code = 'CO' AND term = '#colaboración';

classificator, valoration and write

In [0]:
from pyspark.sql import functions as F

CATALOG = "dbw_taxrisk_dev"
dbutils.widgets.text("country_code", "CO")
COUNTRY_CODE = dbutils.widgets.get("country_code")

# Valuation assumptions (tunable; evaluation measures the error)
LOW, MID, HIGH = 0.9, 1.05, 1.2          # price variation factor
CUES = ["tarjeta de bienvenida"]          # visual cues; TODO: move to config

terms = (spark.table(f"{CATALOG}.config.disclosure_terms")
         .filter(F.col("country_code") == COUNTRY_CODE))
def terms_of(meaning):
    return [r["term"].lower() for r in terms.filter(f"meaning = '{meaning}'").collect()]
paid_terms, inkind_terms, either_terms = terms_of("paid"), terms_of("in_kind"), terms_of("paid_or_in_kind")
print("paid:", paid_terms, "| in_kind:", inkind_terms, "| either:", either_terms)

def has_any(term_list):
    arr = F.array(*[F.lit(t) for t in term_list])
    return F.size(F.array_intersect("hashtags_norm", arr)) > 0

s = spark.table(f"{CATALOG}.silver.posts_enriched")
has_venue = F.col("venue_id").isNotNull()
has_cue = F.lower("image_description").rlike("|".join(CUES))
is_hotel = F.col("venue_category") == "hotel"

# Group-size assumption (guests / nights): not visible in the caption
m_low  = F.lit(1.0)
m_mid  = F.when(is_hotel, 1.5).otherwise(2.0)
m_high = F.when(is_hotel, 2.0).otherwise(3.0)

classified = (s
    .withColumn("pred_class",
        F.when(has_any(paid_terms), "paid")
         .when(has_any(either_terms), "disclosed_collab")
         .when(has_any(inkind_terms), "in_kind")
         .when(has_venue & has_cue, "in_kind")
         .when(has_venue, "unclear")
         .otherwise("organic"))
    .withColumn("class_basis",
        F.when(has_any(paid_terms) | has_any(either_terms) | has_any(inkind_terms), "disclosure_tag")
         .when(has_venue & has_cue, "image_cue")
         .when(has_venue, "venue_mention_no_disclosure")
         .otherwise("no_signal"))
    .withColumn("_needs_value", F.col("pred_class").isin("in_kind", "disclosed_collab", "unclear"))
    .withColumn("value_low_cop",  F.when(F.col("_needs_value"), F.round(F.col("typical_ticket_cop") * m_low  * LOW)))
    .withColumn("value_mid_cop",  F.when(F.col("_needs_value"), F.round(F.col("typical_ticket_cop") * m_mid  * MID)))
    .withColumn("value_high_cop", F.when(F.col("_needs_value"), F.round(F.col("typical_ticket_cop") * m_high * HIGH)))
    .drop("_needs_value"))

(classified.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.posts_classified"))

display(classified.groupBy("pred_class", "class_basis").count().orderBy("pred_class"))

Evaluation